# Notebook 5: Making sense of results. The number you were shown is not the number you will get

**LLMs in Bioinformatics workshop, making sense of results (data leakage, classic vs foundation models)** | ~90 minutes | GPU recommended (free T4), CPU works but is slower

A model paper says "Spearman 0.9". Before you trust it, ask two questions:

1. **Did the model already see the answer?** (data leakage: the test set overlaps with training data, or with the data the foundation model was pre-trained on)
2. **Would something much simpler do just as well?** (a one-hot encoding, a nearest neighbour)

You will answer both on **real protein data**, with a small protein language model (ESM-2, 8 million parameters).

By the end you will be able to:

1. show that **the same model gets very different scores depending on how you split the data**;
2. measure **how similar each test protein is to the training set** and see where the inflated score comes from;
3. compare a **foundation model against simple baselines** under fair and unfair splits;
4. recognise the **kinds of leakage** and audit a claim with a checklist;
5. **check an AI answer against a primary source** and catch a confident hallucination;
6. see the same split lesson on the **workshop's own CCLE data** (predict protein from RNA, for new cell lines and for a new tissue).

### Time plan (90 min)

| Time | Part | What happens |
| --- | --- | --- |
| 0:00-0:08 | **0. Setup** | install, download two real datasets |
| 0:08-0:30 | **1. Same data, five splits** | GB1 protein variants: random vs hard splits, one-hot vs ESM-2 |
| 0:30-0:52 | **2. Who is in your test set?** | measure test-to-train similarity, split by split |
| 0:52-1:10 | **3. Leakage, audit, hallucination** | name the leak, audit a (fictional) abstract, fact-check a confident AI answer against UniProt |
| 1:10-1:25 | **4. The same lesson on CCLE** | protein from RNA: random cell lines vs a held-out tissue |
| 1:25-1:30 | **5. Open time** | challenges, questions, troubleshooting |

Each part: **predict -> try (TODO) -> check cell -> debrief**.

### How to use this notebook

- Run cells top to bottom (**Shift+Enter**). Cells marked **TODO** need `...` replaced with your code (one or two lines).
- **Write your prediction before you run a result cell.** Being wrong is the point: it is how you learn which results to distrust.
- **Try for 3 minutes before opening a hint.**
- Use a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, the ESM-2 steps take several minutes each (they are cached, so you pay once).
- Stuck? See **Troubleshooting** at the bottom, then ask.

## Part 0. Setup (8 min)

Two real datasets from the **FLIP benchmark** (Dallago et al., NeurIPS 2021), downloaded from GitHub:

- **GB1**: 8,733 variants of a small protein domain (protein G, domain B1). Four positions were mutated in all combinations. The measured value is the **binding fitness** to an antibody (IgG-Fc). It was measured in the lab.
- **Meltome (human cell line)**: ~7,000 human proteins with measured **melting temperature** (how hot before the protein falls apart). Measured by proteomics.

In [ ]:
!pip -q install transformers biopython scikit-learn 2>&1 | tail -1

In [ ]:
import io, os, time, zipfile, urllib.request, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
warnings.filterwarnings("ignore")

GB1_URL  = "https://raw.githubusercontent.com/J-SNACKKB/FLIP/main/splits/gb1/splits.zip"
MELT_URL = "https://raw.githubusercontent.com/J-SNACKKB/FLIP/main/splits/meltome/splits.zip"
CACHE_BASE_URL = ""   # instructor may paste a link to emb_gb1.npz / emb_melt.npz here (optional)
SEED = 0

def fetch_zip(url, folder):
    """Download a zip once and unpack it into `folder`."""
    if os.path.isdir(folder) and os.listdir(folder):
        return
    os.makedirs(folder, exist_ok=True)
    data = urllib.request.urlopen(url, timeout=120).read()
    zipfile.ZipFile(io.BytesIO(data)).extractall(folder)

fetch_zip(GB1_URL, "flip_gb1")
fetch_zip(MELT_URL, "flip_melt")
print("GB1 files  :", sorted(f for f in os.listdir("flip_gb1/splits") if f.endswith(".csv")))
print("Meltome files:", sorted(f for f in os.listdir("flip_melt/splits") if f.endswith(".csv")))

## Part 1. Same data, five splits (22 min)

The GB1 data come with **five official ways to split into train and test**. Same 8,733 variants, same labels:

| Split | What the test set contains |
| --- | --- |
| `sampled` | a random 20% of the variants |
| `low_vs_high` | train on low-fitness variants, test on high-fitness ones |
| `one_vs_rest` | train on single mutants, test on everything else |
| `two_vs_rest` | train on variants with up to 2 mutations, test on those with more |
| `three_vs_rest` | train on up to 3 mutations, test on the rest |

**Prediction (write it down, then run):** a simple model is trained on each split. Which split gives the **highest** Spearman correlation, and which the **lowest**?

**My prediction:**
- highest: _______________
- lowest: _______________

In [ ]:
SPLITS = ["sampled", "low_vs_high", "one_vs_rest", "two_vs_rest", "three_vs_rest"]
gb1 = {s: pd.read_csv(f"flip_gb1/splits/{s}.csv") for s in SPLITS}

df = gb1["sampled"]
print(df.shape)
print(df.set.value_counts().to_dict())
print("sequence length:", df.sequence.str.len().unique())

# the 4 mutated positions = the positions where the sequences differ
M = np.array([list(s) for s in df.sequence.values])
SITES = [i for i in range(M.shape[1]) if len(set(M[:, i])) > 1]
print("mutated positions (0-based):", SITES)
print("example variant residues  :", ["".join(M[k, SITES]) for k in range(3)])

### 1a. The simple baseline: one-hot encoding

A **one-hot** encoding says, for each of the 4 mutated positions, *which* of the 20 amino acids is there. No biology, no pre-training. Then a **ridge regression** (a linear model) learns one weight per (position, amino acid).

**TODO 1.** Finish `run_split`: fit the model on the **training rows**, predict the **test rows**, and return Spearman's rho.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"

def one_hot(seqs):
    """4 mutated positions x 20 amino acids = 80 columns of 0/1."""
    X = np.zeros((len(seqs), len(SITES) * 20), dtype="float32")
    for r, s in enumerate(seqs):
        for j, p in enumerate(SITES):
            X[r, j * 20 + AA.index(s[p])] = 1
    return X

def run_split(X, df, alpha=1.0):
    """Fit on train rows, return Spearman rho on test rows."""
    train = (df.set == "train").values
    test = (df.set == "test").values
    y = df.target.values
    model = Ridge(alpha=alpha)
    # TODO 1: fit on the training rows (X[train], y[train]) and predict the test rows (X[test])
    model.fit(..., ...)
    pred = model.predict(...)
    return spearmanr(pred, y[test]).statistic

In [ ]:
# CHECK 1
try:
    _d = gb1["sampled"]
    _rho = run_split(one_hot(_d.sequence.values), _d)
    assert 0.5 < _rho < 1.0, f"rho = {_rho:.2f}: expected a clearly positive value on the random split"
    print(f"OK. One-hot ridge on the random split: rho = {_rho:.3f}")
except (TypeError, ValueError, AttributeError) as e:
    raise AssertionError("TODO 1 not finished: replace each ... with train/test rows of X and y") from e

In [ ]:
rows = []
for s in SPLITS:
    d = gb1[s]
    rows.append({"split": s, "n_train": int((d.set == "train").sum()), "n_test": int((d.set == "test").sum()),
                 "one-hot ridge": run_split(one_hot(d.sequence.values), d)})
res1 = pd.DataFrame(rows).set_index("split")
res1.round(3)

**Debrief question.** Was your prediction right? One model, one dataset, and the score moves by a factor of two or more. Which of these numbers would a paper quote? Which one tells you how the model will do on a **new protein design**?

*Hint to think about:* in a random split, the test variants differ from training variants by a single mutation. In `low_vs_high`, the model must predict values **it has never seen** in training.

### 1b. Now the protein language model

Does ESM-2 fix this? We use the smallest ESM-2 (8M parameters, `facebook/esm2_t6_8M_UR50D`) as a **frozen feature extractor**: feed the sequence, read the hidden state at each of the 4 mutated positions (320 numbers each), and train the **same ridge regression** on top. This is a **linear probe**, as in Notebook 1.

Only the first 60 residues matter here (the GB1 domain and all 4 mutated sites are in it), so we trim to keep it fast.

**Prediction:** will ESM-2 features beat one-hot on the random split? On `low_vs_high`? Write it down.

**My prediction:** random split: ______   low_vs_high: ______

In [ ]:
import torch
from transformers import AutoTokenizer, EsmModel

MODEL = "facebook/esm2_t6_8M_UR50D"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
esm = EsmModel.from_pretrained(MODEL).to(DEVICE).eval()

@torch.no_grad()
def esm_hidden(seqs, batch=64, maxlen=None):
    """Yield (batch_indices, hidden_states, attention_mask) for the sequences."""
    for i in range(0, len(seqs), batch):
        chunk = list(seqs[i:i + batch])
        enc = tokenizer(chunk, return_tensors="pt", padding=True, truncation=maxlen is not None,
                        max_length=maxlen)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        h = esm(**enc).last_hidden_state.float().cpu().numpy()
        yield i, h, enc["attention_mask"].cpu().numpy()

def maybe_download_cache(name):
    if CACHE_BASE_URL and not os.path.exists(name):
        try:
            urllib.request.urlretrieve(CACHE_BASE_URL.rstrip("/") + "/" + name, name)
            print("downloaded cached", name)
        except Exception as e:
            print("cache not available, will compute:", e)

In [ ]:
# Embed every unique GB1 sequence once (all five splits contain the same variants).
GB1_TRIM = 60
all_seqs = sorted(set().union(*[set(d.sequence) for d in gb1.values()]))
print("unique GB1 sequences:", len(all_seqs))

maybe_download_cache("emb_gb1.npz")
if os.path.exists("emb_gb1.npz"):
    z = np.load("emb_gb1.npz", allow_pickle=True)
    gb1_emb = dict(zip(z["seqs"], z["feats"]))
    print("loaded cached GB1 embeddings")
else:
    t0 = time.time()
    trimmed = [s[:GB1_TRIM] for s in all_seqs]
    feats = np.zeros((len(all_seqs), len(SITES) * esm.config.hidden_size), dtype="float32")
    for i, h, mask in esm_hidden(trimmed):
        # token 0 is <cls>, so residue p sits at token p + 1
        feats[i:i + len(h)] = np.concatenate([h[:, p + 1, :] for p in SITES], axis=1)
    gb1_emb = dict(zip(all_seqs, feats))
    np.savez("emb_gb1.npz", seqs=np.array(all_seqs, dtype=object), feats=feats)
    print(f"embedded in {time.time() - t0:.0f} s and saved emb_gb1.npz")

def esm_gb1(seqs):
    return np.stack([gb1_emb[s] for s in seqs])

In [ ]:
res1["ESM-2 8M probe"] = [run_split(esm_gb1(gb1[s].sequence.values), gb1[s], alpha=10.0) for s in SPLITS]
print(res1.round(3))

ax = res1[["one-hot ridge", "ESM-2 8M probe"]].plot.bar(figsize=(8, 3.5), rot=0)
ax.set_ylabel("Spearman rho (test)"); ax.set_title("GB1: same model families, five ways to split")
ax.axhline(0, color="k", lw=0.5); plt.tight_layout(); plt.show()

**Debrief.**

- Compare the two columns *within a row* and *across rows*. Which difference is bigger: model vs model, or split vs split?
- If a paper only reports the `sampled` row, what would you conclude about the language model? Is that fair?
- Fair-comparison rule: **baseline and big model must use the same split, the same labels, the same tuning effort.** Did we tune the ridge `alpha` for both equally? (We did not. Challenge B asks you to.)

## Part 2. Who is in your test set? (22 min)

In Part 1 we *changed the split* and watched the score move. Now we **measure why**: how similar is each test protein to something in the training set?

If a test protein has a close relative (a homolog) in the training set, a model can do well by **remembering the relative's label**, not by understanding anything. That is **homology leakage**.

We use the human **melting temperature** data. FLIP gives an official split in which proteins were first **clustered at 20% sequence identity** and whole clusters were put on one side, so train and test are distant relatives. We will compare it with a **random split of the same proteins**.

In [ ]:
mel = pd.read_csv("flip_melt/splits/human_cell.csv").drop_duplicates("sequence")
mel = mel[(mel.sequence.str.len() >= 60) & (mel.sequence.str.len() <= 400)]   # keep short ones: faster
mel = mel.sample(n=min(2500, len(mel)), random_state=SEED).reset_index(drop=True)
seqs = mel.sequence.values
y = mel.target.values
print(len(mel), "proteins;", "official split:", mel.set.value_counts().to_dict())
print(f"melting temperature: mean {y.mean():.1f} C, sd {y.std():.1f} C")

# Split A: the official cluster split (train/test as given)
idx_train_A = np.where(mel.set == "train")[0]
idx_test_A = np.where(mel.set == "test")[0]
# Split B: random split with the SAME number of test proteins
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(mel))
idx_test_B, idx_train_B = perm[:len(idx_test_A)], perm[len(idx_test_A):]
SPLIT_SETS = {"cluster (official)": (idx_train_A, idx_test_A), "random": (idx_train_B, idx_test_B)}
for k, (a, b) in SPLIT_SETS.items():
    print(f"{k:20s} train {len(a)}  test {len(b)}")

### 2a. Measure similarity to the training set

To find each test protein's **closest training protein** we do two steps:

1. **Shortlist** the 5 most similar training proteins using 3-mer counts (fast, rough).
2. **Align** the test protein with each of the 5 (Smith-Waterman local alignment, Biopython) and keep the best.

The alignment gives a raw score. We turn it into a **similarity between 0 and 1**: divide by the best possible score, which is `2 x length of the shorter sequence` (every residue matches, +2 each). Identical sequence = 1.0. Unrelated proteins land below about 0.1.

**TODO 2.** Write the normalisation.

In [ ]:
from Bio import Align
aligner = Align.PairwiseAligner()
aligner.mode = "local"
aligner.match_score, aligner.mismatch_score = 2, -1
aligner.open_gap_score, aligner.extend_gap_score = -4, -1

kmer_vec = CountVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False)
K3 = normalize(kmer_vec.fit_transform(seqs).astype("float32"))

def best_similarity(test_seq, candidate_seqs):
    """Return (similarity 0..1, index of best candidate)."""
    best, best_j = 0.0, 0
    for j, cand in enumerate(candidate_seqs):
        score = aligner.score(test_seq, cand)
        # TODO 2: divide the score by the best possible score = 2 * length of the SHORTER sequence
        sim = ...
        if sim > best:
            best, best_j = sim, j
    return best, best_j

In [ ]:
# CHECK 2
try:
    s_same, _ = best_similarity(seqs[0], [seqs[0]])
    s_other, _ = best_similarity(seqs[0], [seqs[1]])
    assert abs(s_same - 1.0) < 1e-6, f"a protein against itself should give 1.0, you got {s_same:.3f}"
    assert s_other < 0.5, f"two unrelated proteins should be low, you got {s_other:.3f}"
    print(f"OK. self = {s_same:.2f}, two unrelated proteins = {s_other:.2f}")
except TypeError as e:
    raise AssertionError("TODO 2 not finished: replace the ... with score / (2 * min(len(...), len(...)))") from e

In [ ]:
def nearest_train(idx_train, idx_test, shortlist=5):
    """For each test protein: similarity to its closest training protein, and that protein's label."""
    S = (K3[idx_test] @ K3[idx_train].T).toarray()
    sims, nn_labels = [], []
    for i, row in enumerate(S):
        cand = np.argsort(-row)[:shortlist]
        sim, j = best_similarity(seqs[idx_test[i]], [seqs[idx_train[c]] for c in cand])
        sims.append(sim); nn_labels.append(y[idx_train[cand[j]]])
    return np.array(sims), np.array(nn_labels)

t0 = time.time()
near = {k: nearest_train(a, b) for k, (a, b) in SPLIT_SETS.items()}
print(f"done in {time.time() - t0:.0f} s")

In [ ]:
BINS = [0, 0.1, 0.2, 0.4, 1.01]
LABELS = ["<0.1", "0.1-0.2", "0.2-0.4", ">=0.4"]
comp = pd.DataFrame({k: pd.Series(pd.cut(near[k][0], BINS, labels=LABELS, right=False)).value_counts(normalize=True).reindex(LABELS)
                     for k in near})
print((comp * 100).round(1).astype(str) + " %")
ax = comp.T.plot.bar(stacked=True, figsize=(7, 3.2), rot=0, colormap="viridis")
ax.set_ylabel("share of test set"); ax.set_title("How close is each test protein to the training set?")
ax.legend(title="similarity", bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); plt.show()

**TODO 3.** For each split, compute the **fraction of test proteins with similarity of 0.2 or more** to something in training (`near[k][0]` holds the similarities).

In [ ]:
frac_close = {}
for k in near:
    sims = near[k][0]
    # TODO 3: fraction of test proteins whose similarity is >= 0.2
    frac_close[k] = ...

In [ ]:
# CHECK 3
try:
    assert frac_close["random"] > frac_close["cluster (official)"], "the random split should have MORE close relatives in training"
    print(f"OK. Close relatives in the test set: random {frac_close['random']:.0%} vs cluster split {frac_close['cluster (official)']:.0%}")
except TypeError as e:
    raise AssertionError("TODO 3 not finished: use (sims >= 0.2).mean()") from e

### 2b. Where does the score come from?

Three predictors for melting temperature, same train/test rows:

- **3-mer ridge**: a linear model on 3-mer counts (a classic bag-of-words model).
- **Nearest neighbour (1-NN)**: copy the label of the closest training protein. No learning at all.
- **ESM-2 8M probe**: mean-pooled ESM-2 embedding + ridge.

**Prediction:** the nearest-neighbour "model" copies a label. For which test proteins will it work well? And what will happen to the *aggregate* score when we move from the cluster split to the random split?

**My prediction:** ______________________________________________

In [ ]:
MELT_MAXLEN = 402   # 400 residues + <cls> + <eos>
maybe_download_cache("emb_melt.npz")
if os.path.exists("emb_melt.npz"):
    z = np.load("emb_melt.npz", allow_pickle=True)
    melt_emb = dict(zip(z["seqs"], z["feats"]))
    print("loaded cached Meltome embeddings")
else:
    t0 = time.time()
    feats = np.zeros((len(seqs), esm.config.hidden_size), dtype="float32")
    for i, h, mask in esm_hidden(list(seqs), batch=32, maxlen=MELT_MAXLEN):
        m = mask.copy().astype(float)
        m[:, 0] = 0                                   # drop <cls>
        last = mask.sum(1).astype(int) - 1
        m[np.arange(len(m)), last] = 0                # drop <eos>
        feats[i:i + len(h)] = (h * m[:, :, None]).sum(1) / m.sum(1, keepdims=True)
    melt_emb = dict(zip(seqs, feats))
    np.savez("emb_melt.npz", seqs=np.array(list(seqs), dtype=object), feats=feats)
    print(f"embedded {len(seqs)} proteins in {time.time() - t0:.0f} s and saved emb_melt.npz")
E_ESM = np.stack([melt_emb[s] for s in seqs])

In [ ]:
def per_bin(pred, truth, sims):
    out = {}
    for lab, lo, hi in zip(LABELS, BINS[:-1], BINS[1:]):
        m = (sims >= lo) & (sims < hi)
        out[lab] = spearmanr(pred[m], truth[m]).statistic if m.sum() >= 15 else np.nan
    return out

rows, bins_tab = [], {}
for k, (tr, te) in SPLIT_SETS.items():
    sims, nn = near[k]
    preds = {
        "3-mer ridge": Ridge(alpha=10).fit(K3[tr], y[tr]).predict(K3[te]),
        "1-NN (copy label)": nn,
        "ESM-2 8M probe": Ridge(alpha=10).fit(E_ESM[tr], y[tr]).predict(E_ESM[te]),
    }
    for name, p in preds.items():
        rows.append({"split": k, "model": name, "all test": spearmanr(p, y[te]).statistic})
        bins_tab[(k, name)] = per_bin(p, y[te], sims)
overall = pd.DataFrame(rows).pivot(index="model", columns="split", values="all test")
print("Spearman on the WHOLE test set"); print(overall.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, k in zip(axes, SPLIT_SETS):
    t = pd.DataFrame({name: bins_tab[(k, name)] for name in ["3-mer ridge", "1-NN (copy label)", "ESM-2 8M probe"]})
    t.plot.bar(ax=ax, rot=0); ax.set_title(k); ax.set_xlabel("similarity to closest training protein")
    ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("Spearman rho (this bin only)")
plt.tight_layout(); plt.show()
print("Empty bars = fewer than 15 test proteins in that bin (not enough to say anything).")

**Debrief.**

- Look at the nearest-neighbour bars. For which similarity bin does *copying a label* work, and what does that tell you about a high score on a random split?
- Compare the **whole-test-set** table with the per-bin plot. Is the aggregate score actually higher on the random split? The two test sets contain *different proteins*, so aggregates differ for many reasons. Why does binning by similarity give a cleaner comparison than the overall number?
- Is a better score on the random split evidence that the model "understands" thermostability?
- **Pre-training leakage.** ESM-2 was pre-trained on UniRef50, a clustered snapshot of UniProt (2021). Human proteins like these are almost certainly in it, with sequences (not labels). Our splits above control overlap *between our train and test*, but **not** the overlap between the test set and the model's pre-training data. Published work finds this can still inflate results ([protein LLM pre-training leakage, Meltome thermostability](https://www.biorxiv.org/content/10.1101/2024.07.23.604678.full.pdf); [PPI inference](https://www.biorxiv.org/content/10.1101/2025.04.21.649858.full.pdf)). We cannot retrain ESM-2 here, so we cannot measure it, and that is exactly the problem: **you often can't check**.

## Part 3. Name the leak, audit the claim, catch a hallucination (18 min)

### 3a. The kinds of leakage

| Code | Type | Short description |
| --- | --- | --- |
| `pretrain` | pre-training overlap | the test sequence was in the foundation model's pre-training data |
| `homology` | homology leakage | a close relative of the test protein is in the training set |
| `position` | positional leakage | test and training variants share the same mutated positions |
| `label_memory` | label memorisation | the model has read the labels in its training text (for example, a text LLM that has seen ClinVar classifications in papers) |
| `source` | source/batch leakage | train and test share an experiment, lab or batch with a systematic bias |

**TODO 4.** Match each scenario to a code.

In [ ]:
scenarios = {
    "A": "You test ESM-2 on a protein that is in UniRef50, which ESM-2 was pre-trained on.",
    "B": "Random split of GB1 variants: test variants mutate the same four sites as training variants.",
    "C": "Your test protein shares 70% sequence identity with a training protein.",
    "D": "You ask a text LLM to classify the famous TP53 variant R175H; it has seen its ClinVar entry in papers.",
    "E": "All your train and test proteins were measured in the same lab with the same calibration drift.",
}
# TODO 4: use the codes pretrain / homology / position / label_memory / source
answers = {"A": ..., "B": ..., "C": ..., "D": ..., "E": ...}

In [ ]:
# CHECK 4
truth = {"A": "pretrain", "B": "position", "C": "homology", "D": "label_memory", "E": "source"}
wrong = [k for k in truth if answers.get(k) != truth[k]]
if wrong:
    print("Not yet. Re-read these scenarios:", wrong)
    for k in wrong:
        print(f"  {k}: {scenarios[k]}")
    raise AssertionError("TODO 4: some matches are off")
print("OK. All five matched.")

### 3b. Audit a claim

Below is a **fictional abstract written for this exercise** (not a real paper):

> *"We fine-tuned a 650M-parameter protein language model to predict the stability effect of mutations. On a random 80/20 split of 12,000 variants from 40 proteins, our model reaches Spearman 0.91, far above a one-hot baseline (0.62). These results show that protein language models understand protein stability. Code is available on request."*

**TODO 5.** From the list of possible problems, pick the ones that apply to this abstract and put their keys in `found`.

| Key | Possible problem |
| --- | --- |
| `random_split` | a random split of variants of the same 40 proteins (positional and homology leakage) |
| `weak_baseline` | only a one-hot baseline; no nearest-neighbour or simple sequence-based competitor |
| `no_pretrain_check` | no check for overlap between the 40 proteins and the model's pre-training data |
| `one_seed` | one run, no confidence interval |
| `causal_claim` | "understand" is not supported by a prediction score |
| `gpu_too_small` | the model is too small to be trusted |
| `wrong_metric` | Spearman correlation can never be used for this kind of data |

In [ ]:
# TODO 5: list the keys of the problems you can see in the abstract
found = [...]

In [ ]:
# CHECK 5
real = {"random_split", "weak_baseline", "no_pretrain_check", "one_seed", "causal_claim"}
decoys = {"gpu_too_small", "wrong_metric"}
try:
    f = set(found)
except TypeError:
    raise AssertionError("TODO 5 not finished: put keys as strings in a list")
assert not (f & decoys), f"These are not problems in the abstract: {sorted(f & decoys)}"
assert len(f & real) >= 3, f"Find at least 3 real problems (you have {len(f & real)})"
print(f"OK. You found {len(f & real)} of {len(real)}. Missed: {sorted(real - f) or 'nothing'}")

**Your verdict (2 sentences):** What extra experiment would you ask the authors for before believing the claim? Which split, which baselines?

_______________________________________________

### 3c. Hallucination: a confident answer is not a checked answer

A **hallucination** is output that sounds right and is not supported by anything real: an invented domain, a made-up reference, a score for something the model never saw. It is not random noise. It is *plausible*, which is why it is dangerous.

Below is an **answer about two cancer genes of the CCLE panel (TP53 and KRAS), written for this exercise** to imitate the typical failure. It is **not the output of a specific model** (the live version is Challenge E and Extra A). Read it as if a chat assistant had given it to you:

> *TP53 encodes a protein of 393 amino acids. Residue 175 is an arginine. Residue 248 is a glutamine. Residue 273 is a histidine. In KRAS, residue 12 is a glycine, residue 13 is a glycine and residue 61 is a glutamine. KRAS also carries a protein kinase domain at residues 20 to 180. The TP53 variant R175H is classified pathogenic in ClinVar.*

**Prediction:** every sentence is stated with the same certainty. How many of the **nine** claims do you think are wrong? Write a number: ______

We will fact-check each claim against the **primary source (UniProt)** in code.

In [ ]:
import requests

def fetch_uniprot(acc):
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=30)
    r.raise_for_status()
    d = r.json()
    feats = []
    for f in d.get("features", []):
        loc = f["location"]
        if loc["start"].get("value") and loc["end"].get("value"):
            feats.append({"type": f["type"], "description": f.get("description", ""),
                          "start": loc["start"]["value"], "end": loc["end"]["value"]})
    return {"sequence": d["sequence"]["value"], "features": feats}

try:
    ENTRIES = {"TP53": fetch_uniprot("P04637"), "KRAS": fetch_uniprot("P01116")}
except Exception as e:
    raise RuntimeError("Could not reach UniProt. Wait a minute and re-run this cell.") from e
for g, e in ENTRIES.items():
    print(f"{g}: {len(e['sequence'])} aa; first 40: {e['sequence'][:40]}")

# each sentence of the answer, as a claim we can test
CLAIMS = [
    {"id": 1, "gene": "TP53", "text": "TP53 encodes a protein of 393 amino acids.",          "kind": "length",  "value": 393},
    {"id": 2, "gene": "TP53", "text": "Residue 175 of TP53 is an arginine.",                  "kind": "residue", "pos": 175, "aa": "R"},
    {"id": 3, "gene": "TP53", "text": "Residue 248 of TP53 is a glutamine.",                  "kind": "residue", "pos": 248, "aa": "Q"},
    {"id": 4, "gene": "TP53", "text": "Residue 273 of TP53 is a histidine.",                  "kind": "residue", "pos": 273, "aa": "H"},
    {"id": 5, "gene": "KRAS", "text": "Residue 12 of KRAS is a glycine.",                     "kind": "residue", "pos": 12,  "aa": "G"},
    {"id": 6, "gene": "KRAS", "text": "Residue 13 of KRAS is a glycine.",                     "kind": "residue", "pos": 13,  "aa": "G"},
    {"id": 7, "gene": "KRAS", "text": "Residue 61 of KRAS is a glutamine.",                   "kind": "residue", "pos": 61,  "aa": "Q"},
    {"id": 8, "gene": "KRAS", "text": "KRAS also carries a protein kinase domain at residues 20 to 180.", "kind": "domain", "name": "kinase", "start": 20, "end": 180},
    {"id": 9, "gene": "TP53", "text": "The TP53 variant R175H is classified pathogenic in ClinVar.", "kind": "unchecked"},
]

**TODO 6.** Finish `check_claim`. A claim is **supported** if UniProt agrees, **contradicted** if it does not, and **unchecked** if UniProt cannot tell us (claim 9 needs ClinVar, a different source). `entry` is the UniProt record of the claim's gene.

- **6a (residue):** is the residue at **1-based** position `claim["pos"]` equal to `claim["aa"]`? (Python strings are 0-based.)
- **6b (domain):** is there a feature whose description contains `claim["name"]` (ignore upper/lower case) with exactly `claim["start"]` and `claim["end"]`?

In [ ]:
def check_claim(claim, entry):
    seq = entry["sequence"]
    if claim["kind"] == "unchecked":
        return "unchecked"
    if claim["kind"] == "length":
        ok = len(seq) == claim["value"]
    elif claim["kind"] == "residue":
        ok = ...        # TODO 6a
    elif claim["kind"] == "domain":
        ok = ...        # TODO 6b
    if ok is ...:
        raise NotImplementedError("TODO 6 not finished")
    return "supported" if ok else "contradicted"

In [ ]:
# CHECK 6, then the fact-check table
try:
    verdicts = {c["id"]: check_claim(c, ENTRIES[c["gene"]]) for c in CLAIMS}
except NotImplementedError:
    raise AssertionError("TODO 6 not finished: replace both ... with a True/False expression")
for c in CLAIMS:
    print(f"{verdicts[c['id']]:12s} | {c['text']}")
n_sup = sum(v == "supported" for v in verdicts.values())
n_con = sum(v == "contradicted" for v in verdicts.values())
assert verdicts[8] == "contradicted", "The kinase-domain claim should be contradicted: KRAS has no such feature"
assert verdicts[9] == "unchecked"
assert n_sup >= 4, f"Only {n_sup} claims supported: the true claims fail, so check 1-based positions"
print(f"\nOK. {n_sup} supported, {n_con} contradicted, {sum(v == 'unchecked' for v in verdicts.values())} cannot be checked from this source.")

**Debrief.**

- Every sentence had the **same tone**. Could you have told the wrong ones from the rest without the check? Which is more dangerous: an invented domain (claim 8), or a *real* residue with the *wrong amino acid* (claims 3 and 4)? Notice what claims 3 and 4 have in common: **R248Q** and **R273H** are famous *mutations*. The answer mixed up the mutant with the wild type.
- Claim 9 **cannot be checked against UniProt**. A hallucination detector is only as good as the source you check against: name the right source for each kind of claim (sequence and domains: UniProt; variant classification: ClinVar; papers: PubMed, and the verbatim-quote test of Notebook 2, Part 3).
- A **fluent, specific** answer is evidence of fluency, not of truth. What would you require before using an AI answer in a figure, a grant or a clinical note?

**Hallucination is not only a text problem.** Every tool in this workshop can give a confident, wrong answer. Pick one row and say which **check** you would run.

| Tool | What a "hallucination" looks like | A check you already know |
| --- | --- | --- |
| Text LLM (Notebook 2 Part 3, Extra A) | invented numbers, quotes, IDs, classifications | compare with the primary source; require a verbatim quote |
| ESM-2 probe (Notebook 5, Part 1) | a confident number for a protein unlike anything in training | performance by similarity to training (Part 2) |
| ESMFold (Notebook 4) | a smooth, high-looking structure for a sequence that does not really fold that way | read pLDDT; compare with an experimental structure |
| AlphaGenome (Notebook 3) | a score for a variant whose reference base or build is wrong | check the reference base and the genome build (Notebook 3, Part 2) |

Your own project: which of these will you use, and what is the **source you will check against**? ______________________

## Part 4. The same lesson on CCLE: predict protein from RNA (15 min)

Back to the workshop's data. In Notebook 2 you built a table with, for 19 cancer genes in about 370 cell lines, the **protein level**, the **RNA level** and the **copy number**. A very natural question: *if I only have RNA (cheap), can I predict the protein (expensive)?* The paper found a mean correlation of about 0.48 across genes. Imagine a model that must predict the protein level of a gene in a **new** cell line. *New* can mean two different things:

| Split | The test cell lines are... | The question it answers |
| --- | --- | --- |
| **random cell lines** | any lines, even from a tissue that also appears in training | "can I fill in the gaps in a table I mostly have?" |
| **hold out lineages** | whole tissues (for example all breast lines) **never seen** in training | "will this work for a new tissue, or a new kind of sample?" |

**Prediction:** three predictors: (1) *lineage mean*: copy the average protein level of training lines from the **same tissue**, (2) *own RNA*: a ridge regression on the RNA and copy number of **this gene**, (3) *all-gene RNA*: a ridge regression on the RNA and copy number of **all 20 genes** (more information, and it can tell tissues apart). For each split, which wins? Write it down: random: ____ new tissues: ____

This needs `ccle_multiomics_panel.csv` (the output of Notebook 2, Part 4; your instructor can share it). **No file? Skip to Part 5.** Nothing later depends on it.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


# Class data files: downloaded automatically from the course site (nothing to upload).
import os, requests
CLASS_FILES_URL = "https://raw.githubusercontent.com/AmirAliLoghmani80/CBH-Workshop/main/data/"
def get_class_file(name):
    if os.path.exists(name):
        return name
    if "YOUR-USER" in CLASS_FILES_URL:
        print(f"NOTE: {name} not found and no download address is set. Upload it (left sidebar, Files) or ask the instructor.")
        return None
    r = requests.get(CLASS_FILES_URL + name, timeout=120)
    if r.status_code != 200 or r.content[:15].lstrip().lower().startswith((b"<!doctype", b"<html")):
        print(f"NOTE: could not download {name} (HTTP {r.status_code}). Upload it (left sidebar, Files) or ask the instructor.")
        return None
    with open(name, "wb") as f:
        f.write(r.content)
    print(f"downloaded {name} ({len(r.content)/1e6:.1f} MB)")
    return name

CCLE_FILE = "ccle_multiomics_panel.csv"
get_class_file(CCLE_FILE)

LINEAGES = ["AUTONOMIC_GANGLIA", "BILIARY_TRACT", "BONE", "BREAST", "CENTRAL_NERVOUS_SYSTEM", "CERVIX", "ENDOMETRIUM", "FIBROBLAST",
            "HAEMATOPOIETIC_AND_LYMPHOID_TISSUE", "KIDNEY", "LARGE_INTESTINE", "LIVER", "LUNG", "OESOPHAGUS", "OVARY", "PANCREAS",
            "PLEURA", "PROSTATE", "SALIVARY_GLAND", "SKIN", "SMALL_INTESTINE", "SOFT_TISSUE", "STOMACH", "THYROID",
            "UPPER_AERODIGESTIVE_TRACT", "URINARY_TRACT"]

def lineage_of(cell_line):
    # CCLE names end with the tissue, for example AU565_BREAST. Some tissue names contain underscores, so match the end.
    for L in sorted(LINEAGES, key=len, reverse=True):
        if cell_line.endswith("_" + L):
            return L
    return "OTHER"

ccle = None
if os.path.exists(CCLE_FILE):
    ccle = pd.read_csv(CCLE_FILE)
    need = {"cell_line", "gene", "protein", "rna_log2tpm", "copy_number"}
    if need - set(ccle.columns):
        raise ValueError(f"{CCLE_FILE} lacks the columns {sorted(need - set(ccle.columns))}. It needs the RNA and copy-number layers: use the file built with ccle_panel.csv in Notebook 2.")
    PROT = ccle.pivot_table(index="cell_line", columns="gene", values="protein")
    RNA = ccle.pivot_table(index="cell_line", columns="gene", values="rna_log2tpm").reindex(PROT.index)
    CN = ccle.pivot_table(index="cell_line", columns="gene", values="copy_number").reindex(PROT.index)
    lines = pd.DataFrame({"cell_line": PROT.index, "lineage": [lineage_of(c) for c in PROT.index]})
    print(f"{PROT.shape[0]} cell lines x {PROT.shape[1]} genes; lineages: {lines['lineage'].nunique()}; "
          f"lines with no recognised lineage: {(lines['lineage'] == 'OTHER').sum()}")
    print("protein values present:", f"{PROT.notna().mean().mean():.0%}")
    g_show = "ERBB2" if "ERBB2" in PROT.columns else PROT.columns[0]
    ok_ = PROT[g_show].notna() & RNA[g_show].notna()
    plt.figure(figsize=(4.5, 3.3)); plt.scatter(RNA.loc[ok_, g_show], PROT.loc[ok_, g_show], s=8)
    plt.xlabel(f"{g_show} RNA (log2 TPM)"); plt.ylabel(f"{g_show} protein"); plt.title(f"{g_show}: protein vs RNA"); plt.tight_layout(); plt.show()
else:
    print(f"{CCLE_FILE} not found: Part 4 is skipped. Upload it and re-run this cell.")

### 4a. The two splits

Each split keeps **20%** of the groups for testing. The groups are single cell lines, or whole lineages. We repeat with **5 random seeds**, because one split can be lucky (remember the one-seed problem from 3b).

In [ ]:
SPLIT_KINDS = ["random cell lines", "hold out lineages"]

def make_split(lines, kind, seed, test_frac=0.2):
    groups = np.arange(len(lines)) if kind == "random cell lines" else lines["lineage"].values
    tr, te = next(GroupShuffleSplit(1, test_size=test_frac, random_state=seed).split(lines, groups=groups))
    return set(lines["cell_line"].iloc[tr]), set(lines["cell_line"].iloc[te])

if ccle is not None:
    for kind in SPLIT_KINDS:
        tr, te = make_split(lines, kind, seed=0)
        lin = lines.set_index("cell_line")["lineage"]
        seen = np.mean([lin[c] in set(lin[list(tr)]) for c in te])
        print(f"{kind:20s} train {len(tr):4d}  test {len(te):4d}  | test lines whose tissue is also in training: {seen:.0%}")

**TODO 7.** Write the **lineage-mean predictor**: for each test cell line, return the **mean protein level of the training lines of the same lineage**. For a lineage with no training line, return the mean of all training lines. `train` and `test` are data frames with the columns `lineage` and (in `train`) `protein`; return one prediction per test row, in order.

In [ ]:
def predict_lineage_mean(train, test):
    # TODO 7
    return ...

In [ ]:
# CHECK 7
if ccle is not None:
    _tr = pd.DataFrame({"lineage": ["A", "A", "B", "B", "B"], "protein": [1.0, 3.0, 10.0, 20.0, 30.0]})
    _te = pd.DataFrame({"lineage": ["B", "A", "C"]})
    try:
        _p = np.asarray(predict_lineage_mean(_tr, _te), dtype=float)
    except (TypeError, ValueError):
        raise AssertionError("TODO 7 not finished: return one number per test row")
    assert _p.shape == (3,) and not np.isnan(_p).any(), "one prediction per test row, no NaN"
    assert np.allclose(_p, [20.0, 2.0, 12.8]), f"expected [20, 2, 12.8] (lineage mean, then the overall mean for the unseen lineage C), got {list(_p)}"
    print("OK. Your predictor copies the mean of the same tissue.")

### 4b. The other two predictors, and the comparison

Provided: **own RNA** (the RNA and copy number of the gene being predicted) and **all-gene RNA** (RNA and copy number of all 20 genes). Both feed a ridge regression; missing values are filled with the median of the *training* lines only. The score is the Spearman correlation across the test cell lines of one gene, averaged over genes (0 means no information; a predictor that returns one constant gets 0).

In [ ]:
def rho_of(pred, y):
    return 0.0 if np.std(pred) < 1e-12 else float(spearmanr(pred, y).statistic)

def ridge(alpha=10.0):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=alpha))

N_SEEDS, MIN_TEST = 5, 8
if ccle is not None:
    lin_of = lines.set_index("cell_line")["lineage"]
    all_X = pd.concat([RNA.add_prefix("rna_"), CN.add_prefix("cn_")], axis=1).dropna(axis=1, how="all")
    runs = []
    for kind in SPLIT_KINDS:
        for seed in range(N_SEEDS):
            tr_names, te_names = make_split(lines, kind, seed)
            per_gene = []
            for g in PROT.columns:
                y = PROT[g].dropna()
                tr_i = [c for c in y.index if c in tr_names]; te_i = [c for c in y.index if c in te_names]
                if len(te_i) < MIN_TEST or len(tr_i) < 30:
                    continue
                trf = pd.DataFrame({"lineage": lin_of[tr_i].values, "protein": y[tr_i].values})
                tef = pd.DataFrame({"lineage": lin_of[te_i].values})
                own = [c for c in (f"rna_{g}", f"cn_{g}") if c in all_X.columns]
                row = {"lineage mean": rho_of(predict_lineage_mean(trf, tef), y[te_i].values)}
                for name, cols in [("own RNA", own), ("all-gene RNA", list(all_X.columns))]:
                    m = ridge().fit(all_X.loc[tr_i, cols], y[tr_i])
                    row[name] = rho_of(m.predict(all_X.loc[te_i, cols]), y[te_i].values)
                per_gene.append(row)
            if per_gene:
                runs.append({"split": kind, "seed": seed, "genes": len(per_gene), **pd.DataFrame(per_gene).mean().to_dict()})
    runs = pd.DataFrame(runs)
    cols = ["lineage mean", "own RNA", "all-gene RNA"]
    summary = runs.groupby("split")[cols].agg(["mean", "std"]).reindex(SPLIT_KINDS)
    print("Spearman across test cell lines, averaged over genes; mean (sd) over", N_SEEDS, "seeds")
    print(pd.DataFrame({c: [f"{summary[c]['mean'][k]:.2f} ({summary[c]['std'][k]:.2f})" for k in SPLIT_KINDS] for c in cols}, index=SPLIT_KINDS))
    ax = runs.groupby("split")[cols].mean().reindex(SPLIT_KINDS).plot.bar(figsize=(6.5, 3.4), rot=0)
    ax.set_ylabel("Spearman (test)"); ax.axhline(0, color="k", lw=0.5); plt.tight_layout(); plt.show()

**Debrief.**

- Compare with your prediction. What happened to the **lineage mean** when the whole tissue was held out? A model that only *looks up the tissue* is not a model of the biology, and the random split could not tell you.
- The **all-gene RNA** model can recognise a tissue from the RNA of many genes. Did it keep its advantage over **own RNA** when the tissue was new? If not, what was it really using?
- If the question is "will this work on my new tissue?", which split matches, and which number would you put in a paper?
- Nusinow et al. (2020), the source of the protein data, report a mean protein-RNA correlation of about 0.48 over all genes. Where does your *own RNA* score sit, and what do you not know about how your 20 genes were chosen?
- This used only simple models. What would you have to know about a **foundation model's** training data before trusting a high score here (the same question as Part 2)?

## Part 5. Open time (5 min)

Pick **one** challenge, or bring your own question.

**Challenge A: more labels, who wins? (GB1)**
Pick the `sampled` split. Train both the one-hot ridge and the ESM-2 probe on only 50, 100, 500, 2000 training variants (random subsets of the train rows), evaluate on the whole test set. Plot Spearman vs number of labels. Where do the curves cross, if they do?

**Challenge B: tune both fairly (GB1)**
Try `alpha` in `[0.01, 0.1, 1, 10, 100, 1000]` for **both** models on `low_vs_high`. Choose `alpha` with a validation set taken from the *training* data (never from the test). Does the winner change?

**Challenge C: a stricter similarity (Meltome)**
Re-do Part 2 with `shortlist=20` instead of 5. Do the proportions of close relatives change? Is the 5-candidate shortcut hiding some relatives?

**Challenge D: a harder random split (Meltome)**
Build a *third* split: remove from the random split's training set every protein with similarity >= 0.3 to any test protein. How close does it get to the cluster split?

**Challenge E: live hallucination (needs an LLM API key from Extra A, the literature RAG notebook):** ask a text LLM the same questions as in Part 3c, then for the ClinVar classification of a well-known TP53 variant and of a **made-up** variant. Fact-check each answer with `check_claim` or ClinVar. Ask the same question 3 times: do the answers agree? What does a confident answer for the real variant prove, and what does it not?

## Take-home checklist (copy it for your own projects)

1. **Split like the deployment.** Will the model see new proteins? Then split by *cluster*, not by row. New cell lines of known tissues, or a new tissue? Each is a different question (Part 4).
2. **Report the split next to every number.** A score without its split is not a result.
3. **Measure test-to-train similarity** and report performance by similarity bin.
4. **Always include a do-nothing baseline, a one-hot or k-mer baseline and a nearest-neighbour baseline.**
5. **Same split, same data, same tuning effort for baseline and big model.**
6. **Ask about the foundation model's pre-training data.** If you cannot check, say so, and treat the result as an upper bound.
7. **Several seeds, confidence intervals.**
8. **Claims must match evidence.** Prediction accuracy is not "understanding".

## Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `HTTPError` or timeout in Part 0 | GitHub download blocked or slow | Re-run the cell; ask the instructor for the zip files and upload them to `flip_gb1/` and `flip_melt/` |
| ESM embedding cell takes minutes | CPU runtime | Switch to *T4 GPU* (Runtime menu) or use the cached `.npz` files from the instructor |
| `CUDA out of memory` | batch too big | Restart the runtime; lower `batch=` to 16 in `esm_hidden` |
| `... not finished` messages from CHECK cells | a TODO still has `...` | Replace the `...` with your code and rerun the TODO cell, then the check |
| Different numbers from your neighbour | random seeds, sample size | Expected small differences; the *pattern* should match |
| Part 2 similarity loop is slow | CPU is busy | It should take < 1 min; if it does not, lower the sample size from 2500 to 1500 in the Meltome loading cell |
| Variable not defined | skipped a cell | Run all cells above (Runtime > Run before) |